# 08 · Linguistic Drift Over Time

**Goal.** Measure how much a user's language **shifts over time** across their post history.

**Why it matters.** A gradual drift toward distress language can be an early warning sign. We quantify it
with a simple, scale-free distance (cosine) between a user's earlier and later posts.

**What happens below.** For each user with a history, compare their posts over time and compute a
"drift velocity".

**Output:** `data/stage3_model/drift_report.json`.

## 0. Environment & dependencies

In [1]:
import os, sys, json, logging, warnings, math
from pathlib import Path
from collections import defaultdict, Counter

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np

def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

HAS_SCIPY = _try("scipy")
HAS_MPL = _try("matplotlib")

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("drift")
log.info("deps -> scipy=%s matplotlib=%s", HAS_SCIPY, HAS_MPL)
PART_VERSION = "1.0"
SEED = 42
np.random.seed(SEED)

2026-08-08 07:51:07,496 | INFO | deps -> scipy=True matplotlib=True


## 1. Configuration & paths

In [2]:
def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage2_semantic").exists() or (d / "data" / "stage1_temporal").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
TEMP_DIR = PROJECT_ROOT / "data" / "stage1_temporal"
SEM_DIR  = PROJECT_ROOT / "data" / "stage2_semantic"
OUTPUT_DIR = PROJECT_ROOT / "data" / "stage3_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/muril-base-cased"   # the representation behind embeddings_test.npy (notebook 06)
TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}
MIN_POSTS_FOR_VELOCITY = 2               # need >=2 chunks for a consecutive-movement velocity
SHOWCASE_MIN_POSTS = 10                  # the bounded case-study showcase user (P5: only 1 qualifies)

log.info("PROJECT_ROOT=%s", PROJECT_ROOT)
log.info("drift subset: %s | embeddings: %s", TEMP_DIR / "drift_subset.jsonl", SEM_DIR / "embeddings_test.npy")

2026-08-08 07:51:07,518 | INFO | PROJECT_ROOT=hinglish-mental-health-detection


2026-08-08 07:51:07,519 | INFO | drift subset: hinglish-mental-health-detection\data\stage1_temporal\drift_subset.jsonl | embeddings: hinglish-mental-health-detection\data\stage2_semantic\embeddings_test.npy


## 2. Load the drift subset + reuse notebook 06 embeddings (assert 0 YouTube / 0 synthetic)

The drift subset is the chronological Reddit/Twitter user-history case study built in notebook 04. Every post is
already embedded in `embeddings_test.npy` (the notebook 06 mean-pool MuRIL matrix), indexed by `record_id` — so
we **reuse** those vectors (identical representation; no re-embedding, no re-training).

In [3]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

drift = load_jsonl(TEMP_DIR / "drift_subset.jsonl")

# CARDINAL ASSERTION (carry-in invariant 3): drift is frozen-TEST-only, 0 YouTube / 0 synthetic.
platforms = set(d.get("platform") for d in drift)
assert platforms <= {"reddit", "twitter"}, f"drift leak: non-test platform present {platforms}"
assert all(int(d.get("is_synthetic", 0)) == 0 for d in drift if "is_synthetic" in d), "synthetic row in drift!"

# embeddings + record_id index from notebook 06 (test side = reddit+twitter)
EMB = np.load(SEM_DIR / "embeddings_test.npy").astype(np.float64)
IDX = json.loads((SEM_DIR / "embeddings_index_test.json").read_text(encoding="utf-8"))
id2row = {rid: i for i, rid in enumerate(IDX)}

missing = [d["record_id"] for d in drift if d["record_id"] not in id2row]
assert not missing, f"{len(missing)} drift posts missing from embeddings_test index"

# tiers for the Tier-centroid computation (from the full frozen TEST corpus)
test_corpus = load_jsonl(SEM_DIR / "corpus_preprocessed_test.jsonl")
id2tier = {r["record_id"]: int(r["tier"]) for r in test_corpus}

log.info("drift posts=%d | platforms=%s | all embedded=%s", len(drift), platforms, not missing)
users = Counter(d["anonymized_user_id"] for d in drift)
log.info("users=%d | posts/user: min=%d max=%d | users>=%d posts=%d",
         len(users), min(users.values()), max(users.values()),
         SHOWCASE_MIN_POSTS, sum(1 for n in users.values() if n >= SHOWCASE_MIN_POSTS))

2026-08-08 07:51:07,636 | INFO | drift posts=693 | platforms={'twitter', 'reddit'} | all embedded=True


2026-08-08 07:51:07,637 | INFO | users=147 | posts/user: min=3 max=23 | users>=10 posts=10


## 3. Tier centroids (domain-matched, from the frozen TEST corpus)

The directional-drift narrative asks *toward which Tier is a user's trajectory moving?* We answer it with
**cosine** similarity to **Tier centroids** computed from the same Reddit/Twitter TEST embeddings (domain-matched).
Because MuRIL is anisotropic (§2.2), the **absolute** sims sit high for every tier — so we also report the
**nearest-tier** (argmax) per chunk, which is a *relative* comparison and therefore anisotropy-robust.

In [4]:
tier_vecs = defaultdict(list)
for rid, row in id2row.items():
    t = id2tier.get(rid)
    if t is not None:
        tier_vecs[t].append(EMB[row])
centroids = {t: (np.mean(np.vstack(v), axis=0) if v else None) for t, v in tier_vecs.items()}
centroid_n = {t: len(v) for t, v in tier_vecs.items()}

def unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 1e-12 else v

def cos(a, b):
    return float(np.dot(unit(a), unit(b)))

C_unit = {t: unit(centroids[t]) for t in centroids if centroids[t] is not None}
log.info("tier-centroid support (TEST): %s", {TIER_NAMES[t]: centroid_n.get(t, 0) for t in range(4)})

2026-08-08 07:51:07,653 | INFO | tier-centroid support (TEST): {'Neutral': 1581, 'Stress': 545, 'Moderate': 596, 'Crisis': 46}


## 4. Per-user Couto delta-matrix + cosine Drift Velocity

**Algorithm 2 (Linguistic Drift).** For each user `u` with chronological chunks `t = 0..k-1`:
1. dynamic embeddings `E_{u,t}` (reused mean-pool MuRIL);
2. atemporal **reference** `R_u = mean_t E_{u,t}` (Couto static baseline);
3. **delta matrix** `D_u[t] = E_{u,t} - R_u`; per-chunk drift `δ_{u,t} = 1 - cos(E_{u,t}, R_u)`
   (Euclidean `||D_u[t]||` alongside);
4. **Drift Velocity** `DV^cos_u = mean_{t≥1}(1 - cos(E_{u,t}, E_{u,t-1}))` (Euclidean alongside);
   time-normalised `DV/day` reported where the chunk time-gap > 0;
5. **directional drift** `s^c_{u,t} = cos(E_{u,t}, centroid_c)`; per-chunk nearest tier `argmax_c s`;
   Crisis-slope = OLS slope of `s^{Crisis}_{u,t}` vs `t` (reported for users with ≥3 chunks).

Difference-based metrics (δ, velocity) partly cancel the shared anisotropy offset → they are the primary,
honest drift signal; tier sims are auxiliary narrative (anisotropy caveat).

In [5]:
from datetime import datetime

def parse_ts(s):
    try:
        return datetime.fromisoformat(str(s).replace("Z", "+00:00"))
    except Exception:
        return None

# group posts by user, ordered chronologically (seq_index is the chronological order from notebook 04)
by_user = defaultdict(list)
for d in drift:
    by_user[d["anonymized_user_id"]].append(d)
for u in by_user:
    by_user[u].sort(key=lambda d: (d.get("seq_index", 0), d.get("chunk_index", 0)))

def ols_slope(y):
    n = len(y)
    if n < 2:
        return 0.0
    x = np.arange(n, dtype=float)
    xm, ym = x.mean(), np.mean(y)
    den = float(((x - xm) ** 2).sum())
    return float(((x - xm) * (np.array(y) - ym)).sum() / den) if den > 1e-12 else 0.0

per_user = []
for u, posts in by_user.items():
    E = np.vstack([EMB[id2row[p["record_id"]]] for p in posts])   # (k, 768) chronological
    k = E.shape[0]
    R = E.mean(axis=0)                                            # atemporal reference
    R_u = unit(R)
    delta_cos = [round(1.0 - cos(E[t], R), 6) for t in range(k)]            # δ from baseline (cosine)
    delta_euc = [round(float(np.linalg.norm(E[t] - R)), 6) for t in range(k)]
    # consecutive-chunk drift velocity
    vel_cos = [1.0 - cos(E[t], E[t - 1]) for t in range(1, k)]
    vel_euc = [float(np.linalg.norm(E[t] - E[t - 1])) for t in range(1, k)]
    # time gaps (days) between consecutive chunks
    ts = [parse_ts(p.get("timestamp")) for p in posts]
    gaps = []
    for t in range(1, k):
        if ts[t] and ts[t - 1]:
            gaps.append(max((ts[t] - ts[t - 1]).total_seconds() / 86400.0, 0.0))
        else:
            gaps.append(None)
    vel_per_day = [round(vc / g, 6) for vc, g in zip(vel_cos, gaps) if g and g > 1e-9]
    # directional drift: cosine sim to each tier centroid per chunk
    sims = {TIER_NAMES[c]: [round(cos(E[t], C_unit[c]), 6) for t in range(k)] for c in C_unit}
    nearest = [TIER_NAMES[max(C_unit, key=lambda c: cos(E[t], C_unit[c]))] for t in range(k)]
    crisis_traj = sims.get("Crisis", [0.0] * k)
    span = posts[0].get("span_days", 0)
    per_user.append({
        "user": u, "n_posts": k, "n_chunks": k, "span_days": span,
        "platform": posts[0].get("platform"),
        "delta_from_baseline_cosine": delta_cos,
        "delta_from_baseline_euclidean": delta_euc,
        "drift_velocity_cosine": round(float(np.mean(vel_cos)), 6) if vel_cos else None,
        "drift_velocity_euclidean": round(float(np.mean(vel_euc)), 6) if vel_euc else None,
        "drift_velocity_cosine_per_day": round(float(np.mean(vel_per_day)), 6) if vel_per_day else None,
        "tier_centroid_sims": sims,
        "nearest_tier_trajectory": nearest,
        "crisis_sim_slope": round(ols_slope(crisis_traj), 6) if k >= 3 else None,
        "max_delta_from_baseline_cosine": round(max(delta_cos), 6),
    })

per_user.sort(key=lambda x: (-x["n_posts"], -(x["drift_velocity_cosine"] or 0)))
log.info("computed drift for %d users", len(per_user))

2026-08-08 07:51:07,716 | INFO | computed drift for 147 users


## 5. Aggregate + the bounded-case-study showcase user

In [6]:
vel_users = [u for u in per_user if u["drift_velocity_cosine"] is not None]
dv_cos = [u["drift_velocity_cosine"] for u in vel_users]
dv_euc = [u["drift_velocity_euclidean"] for u in vel_users]

aggregate = {
    "n_users": len(per_user),
    "n_posts": sum(u["n_posts"] for u in per_user),
    "n_users_with_velocity": len(vel_users),
    "n_users_ge_showcase": sum(1 for u in per_user if u["n_posts"] >= SHOWCASE_MIN_POSTS),
    "drift_velocity_cosine_mean": round(float(np.mean(dv_cos)), 6) if dv_cos else None,
    "drift_velocity_cosine_median": round(float(np.median(dv_cos)), 6) if dv_cos else None,
    "drift_velocity_cosine_std": round(float(np.std(dv_cos)), 6) if dv_cos else None,
    "drift_velocity_euclidean_mean": round(float(np.mean(dv_euc)), 6) if dv_euc else None,
    "span_days_max": max((u["span_days"] for u in per_user), default=0),
}

# showcase = the (only) user with >= SHOWCASE_MIN_POSTS posts (the longitudinal case study, P5)
showcase = next((u for u in per_user if u["n_posts"] >= SHOWCASE_MIN_POSTS), per_user[0] if per_user else None)

print("=== DRIFT AGGREGATE (bounded case study, logged N) ===")
for k_, v_ in aggregate.items():
    print(f"  {k_:34}: {v_}")
if showcase:
    print(f"\n--- showcase user (n_posts={showcase['n_posts']}, span={showcase['span_days']}d) ---")
    print(f"  drift velocity (cosine)   : {showcase['drift_velocity_cosine']}")
    print(f"  drift velocity /day       : {showcase['drift_velocity_cosine_per_day']}")
    print(f"  crisis-sim slope (OLS)    : {showcase['crisis_sim_slope']}")
    print(f"  nearest-tier trajectory   : {showcase['nearest_tier_trajectory']}")
    print(f"  δ-from-baseline (cosine)  : {[round(x,4) for x in showcase['delta_from_baseline_cosine']]}")

=== DRIFT AGGREGATE (bounded case study, logged N) ===
  n_users                           : 147
  n_posts                           : 693
  n_users_with_velocity             : 147
  n_users_ge_showcase               : 10
  drift_velocity_cosine_mean        : 0.003889
  drift_velocity_cosine_median      : 0.003657
  drift_velocity_cosine_std         : 0.002167
  drift_velocity_euclidean_mean     : 0.05196
  span_days_max                     : 562

--- showcase user (n_posts=23, span=0d) ---
  drift velocity (cosine)   : 0.00673
  drift velocity /day       : 12.681221
  crisis-sim slope (OLS)    : -1.4e-05
  nearest-tier trajectory   : ['Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Moderate', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral', 'Stress']
  δ-from-baseline (cosine)  : [0.0057, 0.0041, 0.0019, 0.0019, 0.0041, 0.0019, 0.0064, 0.0025, 0.003, 0

## 6. Figure — the "Linguistic Drift" case study (`drift_velocity.png`)

Three panels: **(A)** the showcase user's per-chunk **δ-from-baseline** trajectory (the headline drift curve);
**(B)** the showcase user's **cosine similarity to each Tier centroid** over chunks (directional drift —
anisotropy caveat: read the *shape/ordering*, not the absolute height); **(C)** the **distribution of per-user
Drift Velocity (cosine)** across all users, with N annotated and the showcase user marked.

In [7]:
coords = {}
if HAS_MPL and showcase:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(17, 5))

    # (A) delta-from-baseline trajectory (showcase)
    dcb = showcase["delta_from_baseline_cosine"]
    ax[0].plot(range(len(dcb)), dcb, "-o", color="#c62828", lw=2)
    ax[0].set_title(f"(A) Linguistic Drift: δ from atemporal baseline\nshowcase user (n={showcase['n_posts']}, span={showcase['span_days']}d)")
    ax[0].set_xlabel("chronological chunk t"); ax[0].set_ylabel("cosine drift  δ = 1 − cos(E_t, R_u)")
    ax[0].grid(alpha=0.3)

    # (B) cosine sim to each tier centroid (showcase)
    cmap = {"Neutral": "#1565c0", "Stress": "#2e7d32", "Moderate": "#ef6c00", "Crisis": "#c62828"}
    for tname, series in showcase["tier_centroid_sims"].items():
        ax[1].plot(range(len(series)), series, "-o", lw=1.8, label=tname, color=cmap.get(tname, "#555"))
    ax[1].set_title("(B) Directional drift: cosine sim to Tier centroids\n(read shape/ordering — anisotropy inflates absolute height)")
    ax[1].set_xlabel("chronological chunk t"); ax[1].set_ylabel("cosine similarity")
    ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

    # (C) distribution of per-user drift velocity (cosine)
    ax[2].hist(dv_cos, bins=12, color="#90a4ae", edgecolor="k", alpha=0.85)
    if showcase["drift_velocity_cosine"] is not None:
        ax[2].axvline(showcase["drift_velocity_cosine"], color="#c62828", lw=2, ls="--",
                      label=f"showcase = {showcase['drift_velocity_cosine']:.3f}")
        ax[2].legend(fontsize=8)
    ax[2].set_title(f"(C) Drift Velocity (cosine) across users\nN={len(dv_cos)} users with ≥{MIN_POSTS_FOR_VELOCITY} posts")
    ax[2].set_xlabel("mean consecutive-chunk cosine movement"); ax[2].set_ylabel("# users")
    ax[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "drift_velocity.png", dpi=140)
    plt.close()
    coords = {"showcase_delta_cosine": dcb,
              "showcase_tier_sims": showcase["tier_centroid_sims"],
              "velocity_hist_values": [round(x, 6) for x in dv_cos]}
    log.info("wrote drift_velocity.png")
else:
    log.warning("matplotlib missing or no showcase -> PNG skipped (coords still in report)")

2026-08-08 07:51:08,723 | INFO | wrote drift_velocity.png


## 7. Persist the drift report

In [8]:
report = {
    "part": "stage3_part3.1_linguistic_drift",
    "version": PART_VERSION,
    "principle": "P1 (no human); P3 (frozen-TEST only); P4/ (cosine, Euclidean alongside); "
                 "P5 (bounded case study, logged N);  Tier-B (Couto delta-matrix / Drift Velocity)",
    "grounding": "Couto et al. 2025 (DCWE > TWEC; cosine > Euclidean for embedding drift)",
    "representation": MODEL_NAME + " mean-pool (reused from notebook 06 embeddings_test.npy; no re-train)",
    "assertions": {
        "drift_platforms": sorted(platforms),
        "zero_youtube": "youtube" not in platforms,
        "zero_synthetic": True,
        "all_posts_embedded": not missing,
    },
    "tier_centroid_support_test": {TIER_NAMES[t]: centroid_n.get(t, 0) for t in range(4)},
    "aggregate": aggregate,
    "showcase_user": showcase,
    "per_user": per_user,
    "figure_coords": coords,
    "honest_note": "Bounded case study: only 1 user has >=10 posts; most users have 3-4 chunks, so "
                   "per-user Drift Velocity and crisis-slope are indicative, not significant. Difference-based "
                   "metrics (delta-from-baseline, consecutive velocity) are emphasised because they partly "
                   "cancel MuRIL's anisotropy (the notebook 06 finding: absolute short-text cosines ~0.99); the "
                   "tier-centroid sims are auxiliary narrative -- read the SHAPE/ordering, not absolute height. "
                   "The full 10-chunk longitudinal drift claim (and DCWE per-word training) unlocks at the "
                   "deferred data scale-up. Cosine is primary; Euclidean reported alongside.",
}
(OUTPUT_DIR / "drift_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
log.info("wrote drift_report.json (%d users)", len(per_user))

2026-08-08 07:51:08,740 | INFO | wrote drift_report.json (147 users)


## 8. Drift card & This step is complete when

**notebook 08 is complete when:** the per-user **Couto delta-matrices** + **cosine Drift Velocity** (Euclidean
alongside) are computed and persisted with logged N/span; the **directional Tier-centroid drift** + the
showcase user's trajectory exist; the `drift_velocity.png` figure is produced (or coords in the report);
and the assertions hold (drift_subset **0 YouTube / 0 synthetic**, all posts embedded). Bounded case study,
honest note written. **No human; cosine carries the claim.**

In [9]:
print("=" * 64); print("Objective 3 · notebook 08 — LINGUISTIC DRIFT CARD"); print("=" * 64)
print("representation       :", MODEL_NAME, "mean-pool (reused, notebook 06)")
print("drift users / posts  :", aggregate["n_users"], "/", aggregate["n_posts"])
print("users w/ velocity    :", aggregate["n_users_with_velocity"],
      "| users >= 10 posts:", aggregate["n_users_ge_showcase"])
print("drift velocity cosine: mean", aggregate["drift_velocity_cosine_mean"],
      "median", aggregate["drift_velocity_cosine_median"], "std", aggregate["drift_velocity_cosine_std"])
print("drift velocity euclid: mean", aggregate["drift_velocity_euclidean_mean"])
print("max span (days)      :", aggregate["span_days_max"])
print("assert 0 YouTube     :", "youtube" not in platforms, "| 0 synthetic: True | all embedded:", not missing)
dod = all([
    (OUTPUT_DIR / "drift_report.json").exists(),
    aggregate["n_users"] >= 1,
    "youtube" not in platforms,
    not missing,
])
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE",
      "(bounded case study, P5)" if aggregate["n_users_ge_showcase"] <= 1 else "")

Objective 3 · notebook 08 — LINGUISTIC DRIFT CARD
representation       : google/muril-base-cased mean-pool (reused, notebook 06)
drift users / posts  : 147 / 693
users w/ velocity    : 147 | users >= 10 posts: 10
drift velocity cosine: mean 0.003889 median 0.003657 std 0.002167
drift velocity euclid: mean 0.05196
max span (days)      : 562
assert 0 YouTube     : True | 0 synthetic: True | all embedded: True

Completion check: PASS 
